# Chapter 3 · Structured outputs

Extract from each e-mail the customer's name, the order number, the product and the category, in a JSON object that a program can read: five ways of asking for it, validation with Pydantic, retries and a template with constrained generation.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. With a GPU, the notebook uses Qwen2.5-1.5B-Instruct; without one, Qwen3-0.6B. The whole notebook takes a few minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. The model and the data

The first time, the model and the file with the 24 e-mails and their correct data are downloaded.

In [ ]:
"""Chapter 3: structured outputs. Open model, on GPU or CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path
from typing import Literal, Optional

import torch
from pydantic import BaseModel, ConfigDict, ValidationError, constr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
MODEL_CPU = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose based on the environment
DATA = Path("data")
EMAILS = DATA / "extraction_emails.json"
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None):
    """Loads the model and its tokenizer on the GPU if there is one; otherwise, CPU."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = MODEL_GPU if device == "cuda" else MODEL_CPU
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    model.to(device).eval()
    print(f"Model {name} on {device}")
    return tok, model


def load_emails():
    """The e-mails with their correct data; in Colab, they are downloaded."""
    if not EMAILS.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + EMAILS.name, EMAILS)
    return json.loads(EMAILS.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)
emails = load_emails()
print(len(emails), "e-mails. The first one:")
print(json.dumps(emails[0], ensure_ascii=False, indent=1))

## 2. Chatting

`respond` receives the whole conversation, because the retry needs to add turns to it.

In [ ]:
def messages(system, user, examples=()):
    """System message, solved examples (question-answer pairs) and the question."""
    conversation = [{"role": "system", "content": system}] if system else []
    for question, answer in examples:
        conversation.append({"role": "user", "content": question})
        conversation.append({"role": "assistant", "content": answer})
    return conversation + [{"role": "user", "content": user}]


def conversation_text(conversation):
    """The conversation with the chat template, ready for the model to answer."""
    return tok.apply_chat_template(conversation, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)


def respond(conversation, max_tokens=200):
    """The model's answer to a conversation, with temperature 0."""
    inputs = tok(conversation_text(conversation),
                 return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    answer = tok.decode(new, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()

## 3. The schema and validation

In [ ]:
CATEGORIES = ("order", "return", "fault", "inquiry", "complaint")


class Extraction(BaseModel):
    """The data we want from each e-mail."""
    model_config = ConfigDict(extra="forbid")      # no extra keys
    name: Optional[str]
    order: Optional[constr(pattern=r"^NC-\d{5}$")]
    product: Optional[str]
    category: Literal[CATEGORIES]


def read_json(text):
    """The JSON object in the answer, even if it comes with text around it."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end < start:
        raise ValueError("the answer does not contain any JSON object")
    return json.loads(text[start:end + 1])


REASONS = {"missing": "this field is missing",
           "extra_forbidden": "this field is not allowed",
           "string_pattern_mismatch": "it must have the format NC-12345",
           "literal_error": "it must be one of: " + ", ".join(CATEGORIES),
           "string_type": "it must be a string or null"}


def errors(exception):
    """The validation errors, explained in plain English."""
    if isinstance(exception, ValidationError):
        return "; ".join(f'"{e["loc"][0]}": {REASONS.get(e["type"], e["msg"])}'
                         for e in exception.errors())
    return f"it is not valid JSON ({exception})"

Try the validation with data written by hand: one correct and one with three errors.

In [ ]:
print(Extraction(name="Laura", order="NC-10482", product="bike", category="return"))
try:
    Extraction(name="Laura", order="10482", product="bike", category="exchange", urgent=True)
except ValidationError as e:
    print(errors(e))

## 4. Ask, validate and retry

In [ ]:
ASK_JSON = ("From this e-mail sent by a customer of North Cycles, a bike store, "
            "extract the customer's name, the order number, the product and the "
            "category of the e-mail. Answer in JSON.")


SCHEMA = (
    "Extract the data from an e-mail sent by a customer of North Cycles, a bike "
    "store. Answer only with a JSON object, with no text around it, with exactly "
    "these four keys:\n"
    '- "name": the customer\'s name as they sign the e-mail, or null if they '
    "don't give it.\n"
    '- "order": the order number in the format NC-12345, or null.\n'
    '- "product": the product the e-mail is about, in a few words, or null.\n'
    '- "category": one of these five:\n'
    "  order (tracking or changes to an order already placed),\n"
    "  return (returning or exchanging a product, or its refund),\n"
    "  fault (a product does not work or is broken),\n"
    "  inquiry (questions before buying),\n"
    "  complaint (complaints about how the customer was treated or the service).\n"
    "If a piece of data does not appear in the e-mail, write null: do not make it up.")


EXAMPLES = [
    ("Hi, the saddle from order 10230 cracked after a month. Can you replace "
     "it? Regards, Rose Dean",
     '{"name": "Rose Dean", "order": "NC-10230", "product": "saddle", '
     '"category": "fault"}'),
    ("I've been waiting two days for an answer from you. This is unacceptable.",
     '{"name": null, "order": null, "product": null, '
     '"category": "complaint"}')]

In [ ]:
def extract(email, system, examples=(), retries=0):
    """Asks for the data and, if they are not valid, asks again with the errors."""
    conversation = messages(system, email, examples)
    for attempt in range(retries + 1):
        answer = respond(conversation)
        try:
            return answer, Extraction(**read_json(answer)), attempt
        except (ValueError, TypeError, ValidationError) as e:
            conversation += [
                {"role": "assistant", "content": answer},
                {"role": "user", "content": f"Your answer is not valid: "
                 f"{errors(e)}. Fix it and answer only with the JSON."}]
    return answer, None, attempt

In [ ]:
email = emails[3]["email"]
print(email, "\n")
print("Ask for JSON:\n", extract(email, ASK_JSON)[0], "\n")
print("With schema and examples:\n", extract(email, SCHEMA, EXAMPLES)[0])

## 5. Fill in a template (constrained generation)

The program writes the structure and the model only the values, with limited options.

In [ ]:
def ids(text):
    """The token numbers of a text, without special tokens."""
    return tok(text, add_special_tokens=False)["input_ids"]


def choose(prefix, options):
    """The most probable option as a continuation of the prefix."""
    base = ids(prefix)
    scores = []
    for option in options:
        inputs = torch.tensor([base + ids(option)], device=model.device)
        with torch.no_grad():
            logp = torch.log_softmax(model(inputs).logits[0, :-1], dim=-1)
        positions = range(len(base) - 1, inputs.shape[1] - 1)
        scores.append(sum(logp[i, inputs[0, i + 1]].item() for i in positions))
    return options[scores.index(max(scores))]


def write(prefix, allowed=None, max_tokens=15):
    """Generates free text up to the closing quote, or only with certain tokens."""
    inputs = torch.tensor([ids(prefix)], device=model.device)
    limit = (lambda batch, generated: allowed) if allowed else None
    with torch.no_grad():
        output = model.generate(inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefix_allowed_tokens_fn=limit,
                                pad_token_id=tok.eos_token_id)
    return tok.decode(output[0, inputs.shape[1]:]).split('"')[0]


def fill_in(email):
    """The program writes the structure; the model, only the values."""
    text = conversation_text(messages(SCHEMA, email, EXAMPLES))
    data = {}
    for field in ("name", "order", "product", "category"):
        text += ("{" if field == "name" else ", ") + f'"{field}":'
        if field == "category":
            value = choose(text, [f' "{c}"' for c in CATEGORIES])[2:-1]
        elif choose(text, [" null", ' "']) == " null":
            value = None
        elif field == "order":
            digits = [ids(str(d))[0] for d in range(10)]
            value = "NC-" + write(text + ' "NC-', digits, max_tokens=5)
        else:
            value = write(text + ' "').strip()
        data[field] = value
        text += " null" if value is None else f' "{value}"'
    return json.dumps(data, ensure_ascii=False), Extraction(**data), 0

In [ ]:
print(fill_in(emails[3]["email"])[0])

## 6. Compare with the correct data

In [ ]:
def strip_accents(text):
    """Lowercase and without accents, to compare without worrying about them."""
    return "".join(c for c in unicodedata.normalize("NFD", text.lower().strip())
                   if unicodedata.category(c) != "Mn")


def is_correct(field, obtained, expected):
    """Does a field match? For the product, containing the keyword is enough."""
    if expected is None or obtained in (None, "", "null"):
        return expected is None and obtained in (None, "", "null")
    obtained, expected = strip_accents(str(obtained)), strip_accents(expected)
    return expected in obtained if field == "product" else obtained == expected


FIELDS = ("name", "order", "product", "category")


def evaluate(emails, way):
    """Extracts the data from all the e-mails and counts what comes out right."""
    rows = []
    for e in emails:
        answer, data, attempts = way(e["email"])
        try:
            parsed = json.loads(answer)
            parses = True
        except ValueError:
            parses = False
            try:
                parsed = read_json(answer)
            except ValueError:
                parsed = {}
        if not isinstance(parsed, dict):
            parsed = {}
        row = {"answer": answer, "parses": parses, "valid": data is not None,
               "retries": attempts}
        row.update({k: k in parsed and is_correct(k, parsed[k], e[k]) for k in FIELDS})
        rows.append(row)
    return rows

In [ ]:
WAYS = {
    "1. Ask for JSON": lambda email: extract(email, ASK_JSON),
    "2. With schema": lambda email: extract(email, SCHEMA),
    "3. With examples": lambda email: extract(email, SCHEMA, EXAMPLES),
    "4. Validate and retry": lambda email: extract(email, SCHEMA, EXAMPLES,
                                                   retries=2),
    "5. Fill in template": fill_in,
}


def table(results):
    """One row per way: JSON that parses as is, valid answers and correct fields."""
    columns = ("parses", "valid") + FIELDS
    print(f"{'Way':25s}" + "".join(f"{c[:9]:>10s}" for c in columns))
    for way, rows in results.items():
        print(f"{way:25s}" + "".join(f"{sum(r[c] for r in rows):>7d}/{len(rows)}"
                                     for c in columns))

In [ ]:
results = {}
for way, function in WAYS.items():
    results[way] = evaluate(emails, function)
    print(way, "done")
table(results)

Read the answers in which the best way gets some field wrong:

In [ ]:
for row, email in zip(results["5. Fill in template"], emails):
    mistakes = [field for field in FIELDS if not row[field]]
    if mistakes:
        print(mistakes, row["answer"])
        print("   correct:", {field: email[field] for field in FIELDS})

## Exercises

1. Explain in your own words the difference between reading an answer, validating it and checking that it is correct. Which tool of the chapter takes care of each one?

2. Write by hand three answers that `json.loads` cannot read and three that can be read but do not meet the schema. Check with `Extraction` which errors each one gives.

3. Add to the schema a field `urgent`, of type `bool`, that says whether the customer asks for a quick solution or is angry. How would you describe it in the instruction? And in the template?

4. In the third way, remove the second example, the e-mail with no data. Do the made-up names come back?

5. Change `extract` so that the retry message also includes the fields that were right and asks the model not to change them. Are Anna's name and order kept now?

6. The program normalizes order numbers by asking the model for the format `NC-12345`. Write a Python function that does it from what the customer writes (`10377`, `nc-11020`, `NC 10655`…) and compare it with the model. Which is more reliable? When would you use each one?

7. In `fill_in`, the name is written freely. Restrict its tokens to those that appear in the e-mail, with `prefix_allowed_tokens_fn`. Does it improve? What problems show up?

8. Measure how long each way takes per e-mail. Is the time of the template worth it compared with validating and retrying?

9. Add five harder test e-mails: two products in the same e-mail, a signature with a nickname, or an order number mistyped by the customer (`NC-1037`). What should the model answer in each case? Does it?

10. **Customer record.** Use the best way to extract the data from the 40 test e-mails of chapter 2 and compare the extracted category with the one from that chapter. Is it better to classify separately or together with the other data?

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) and [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), by Qwen (Alibaba), under the Apache 2.0 license. [Pydantic](https://docs.pydantic.dev). Constrained generation: [Willard and Louf (2023)](https://arxiv.org/abs/2307.09702). The North Cycles store and its e-mails are fictional. Original code and experiments.